# 검색 근거를 확인하고 제한된 범위에서 답변하기

08번은 07번의 검색 결과를 사용합니다. 다만 검색 순위가 높다고 바로 답하지 않습니다. 확인한 부품 이름·표 이미지 설명·원문 문장과 질문 범위가 맞을 때만 답합니다.

외부 생성 API나 별도 생성 모델을 사용하지 않습니다. 답변 문구는 정해 둔 형식에 확인한 자료를 넣어 만듭니다. 이것은 **지원하는 질문이 정해진 규칙 기반 실험**이며, 모든 질문의 근거를 자동으로 판별하는 기능은 아닙니다.

이번 자료는 원문 묶음 5개입니다. '확인하기 어렵다'는 전체 PDF에 답이 없다는 뜻이 아니라, 현재 자료와 규칙으로 확인하지 못했다는 뜻입니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 검색 근거를 확인한 제한된 질문에만 정해 둔 문구로 답하는 개인 규칙 실험입니다.
- 추가: 안내도·표·안전벨트 근거 조건을 직접 작성합니다. 외부 생성 API나 새로운 답변 생성 모델은 사용하지 않습니다.
- 현재 .py는 검색 근거 반환까지 옮겼으며 이 노트북의 제한된 답변 실험은 별도 기록입니다.

전체 목록: [수업 기반·추가 적용 기록](../src/car_search_rag/zzong_santafe_lag/learning_additions.md)


### 1. 검색 준비

07번 코드 셀을 실행해 기존 자료와 하이브리드 검색 함수를 메모리에 준비합니다. 파일·Supabase 저장은 실행하지 않습니다.

In [1]:
# [프로젝트 추가 기록] 검색 근거를 확인한 제한된 질문에만 정해 둔 문구로 답하는 개인 규칙 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import contextlib
import io
import re

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
source_path = project_folder / "notebooks" / "07_hybrid_search_comparison.ipynb"
notebook = json.loads(source_path.read_text(encoding="utf-8"))
previous = {}
with contextlib.redirect_stdout(io.StringIO()):
    for index, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] == "code":
            exec(compile("".join(cell["source"]), f"{source_path.name}:cell_{index}", "exec"), previous)

hybrid_search = previous["hybrid_search"]
normalize_keyword = previous["normalize_keyword"]
parent_lookup = previous["parent_lookup"]
print("현재 원문 묶음:", len(parent_lookup))
print("답변 방법:", "확인한 자료를 사용하는 제한된 형식")

현재 원문 묶음: 5
답변 방법: 확인한 자료를 사용하는 제한된 형식


### 2. 질문 범위와 근거 형식 정하기

아래 단어 목록은 질문에 사용할 수 있는 표현을 정합니다. 목록 밖의 내용이 섞이면 답변을 보류합니다. 예를 들어 가격·교체 주기·사용 가능 온도처럼 원문에서 확인하지 않은 내용을 단어 일치만으로 답하지 않습니다.

안내도는 번호·위치 안내·참조 쪽수만, 표는 표시된 점도와 온도 눈금만, 안전벨트 예제는 특정 원문 문장만 답합니다. 숫자나 설명을 임의로 추가하지 않습니다.

In [2]:
# [프로젝트 추가 기록] 검색 근거를 확인한 제한된 질문에만 정해 둔 문구로 답하는 개인 규칙 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
# 공통 문법 표현과 질문 목적을 구분합니다. 다른 유형의 목적을 허용하지 않습니다.
grammar_words = [
    "은", "는", "이", "가", "을", "를", "의", "에", "에서", "과", "와", "으로",
    "인가요", "있나요", "되나요", "알려줘", "알려주세요", "보여줘", "보여주세요", "찾아줘",
]
question_words = grammar_words + [
    "은", "는", "이", "가", "을", "를", "의", "에", "에서", "과", "와", "으로",
    "무엇", "어디", "몇번", "몇쪽", "쪽수", "인가요", "있나요", "되나요",
    "알려줘", "알려주세요", "보여줘", "보여주세요", "찾아줘",
    "위치", "안내도", "번호", "참조", "매뉴얼", "차량", "내부",
]

def fits_vocabulary(question, allowed_words):
    """허용한 표현으로만 구성된 질문인지 확인합니다. 알 수 없는 표현이 남으면 False입니다."""
    alternatives = sorted({normalize_keyword(word) for word in allowed_words if word}, key=len, reverse=True)
    pattern = "(?:" + "|".join(re.escape(word) for word in alternatives) + ")+"
    return re.fullmatch(pattern, normalize_keyword(question)) is not None

def citation(parent):
    """답변의 출처를 PDF 쪽수와 매뉴얼 쪽수로 구분해 기록합니다."""
    return {
        "record_id": parent["metadata"]["record_id"],
        "title": parent["metadata"]["title"],
        "pdf_page_numbers": parent["metadata"]["source_pages"],
        "manual_page_number": parent["metadata"]["manual_page_number"],
        # 앞서 확인한 샘플 페이지의 매뉴얼 쪽수만 기록합니다.
        "manual_page_numbers": [{33: 28, 34: 29, 35: 30, 36: 31, 44: 39, 54: 49}[n]
                                for n in parent["metadata"]["source_pages"]],
    }

def supported_response(parent, answer, evidence, evidence_kind, warning=""):
    """확인한 근거, 출처, 그림 참조를 답변 형식으로 묶습니다."""
    return {
        "status": "supported",
        "answer": answer,
        "evidence_text": evidence,
        "evidence_kind": evidence_kind,
        "warning_text": warning,
        "sources": [citation(parent)],
        "image_parts": parent["image_parts"],
        "reference_hints": [],
    }

# 문장은 실제 원문에서 찾아 쓰며, 원문의 의미를 요약하지 않습니다.
prose_rules = [
    {
        "subject": "안전벨트 스토퍼",
        "pattern": r"안전벨트 스토퍼:.*?\.",
        "words": ["안전벨트", "스토퍼", "쓰면", "사용하면", "어떤", "문제", "왜", "위험", "생기나요"],
    },
    {
        "subject": "차단 클립",
        "pattern": r"차단 클립:.*?\.",
        "words": ["차단클립", "버클", "꽂으면", "경고등", "경고음", "어떻게", "어떤", "문제", "생기나요"],
    },
]

### 3. 검색 결과를 보고 답변 가능 여부 결정

세 검색 후보 안에서 질문에 맞는 자료를 확인합니다. 부품 이름을 찾았더라도 조작 방법을 묻는 질문이면 답하지 않습니다.

표의 0W-20과 온도 눈금은 이전 단계에서 사람이 이미지로 확인한 설명에서 읽습니다. 검색용 설명이 사람의 확인 결과라는 점도 함께 기록합니다. 안전벨트 문장을 답할 때는 같은 원문의 경고 부분을 함께 보관합니다.

In [3]:
# [프로젝트 추가 기록] 검색 근거를 확인한 제한된 질문에만 정해 둔 문구로 답하는 개인 규칙 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
def answer_from_parent(question, parent):
    """한 검색 후보에서 지원하는 질문과 확인한 근거가 맞는 경우에만 답을 만듭니다."""
    kind = parent["metadata"]["content_type"]
    normalized_question = normalize_keyword(question)

    if kind == "vehicle_overview_navigation":
        location_interest = any(term in normalized_question for term in ["위치", "어디", "몇번", "번호", "안내도", "몇쪽", "쪽수", "참조"])
        if not location_interest:
            return None
        # 이번 예제는 한 부품의 정확한 이름을 사용한 위치·번호·쪽수 질문을 지원합니다.
        for item in parent["metadata"]["navigation_items"]:
            name = item["name"]
            if normalize_keyword(name) not in normalized_question:
                continue
            if not fits_vocabulary(question, question_words + [name]):
                continue
            pages = ", ".join(str(n) + "쪽" for n in item["target_manual_pages"])
            evidence = f'{item["number"]}: {name} — 매뉴얼 {pages}'
            assert evidence in parent["content"]
            answer = f'{parent["metadata"]["title"]} 안내도의 {name} 항목은 {item["number"]}번입니다. 항목의 참조 쪽수는 매뉴얼 {pages}입니다.'
            return supported_response(parent, answer, evidence, "verified_navigation_item")

    elif kind == "table_with_images":
        descriptions = {p["name"]: p.get("description") or "" for p in parent["image_parts"]}
        oil_match = re.search(r"1\.6 T-GDi HEV 행에 ([A-Z0-9-]+)이 표시되어 있다", descriptions.get("I2.jpg", ""))
        scale_match = re.search(r"온도 눈금은 (-?\d+)도부터 (-?\d+)도까지", descriptions.get("I1.jpg", ""))
        table_words = grammar_words + [
            "1.6 T-GDi HEV", "엔진오일", "SAE", "점도", "표", "표시된", "표시", "적힌",
            "온도", "눈금", "범위", "부터", "까지", "몇도", "낮은", "높은", "0W-20",
            "무엇", "어디부터", "어디까지",
        ]
        if not fits_vocabulary(question, table_words):
            return None
        if "눈금" in normalized_question or all(term in normalized_question for term in ["표", "온도", "범위"]):
            if scale_match is None:
                return None
            answer = f'표의 온도 눈금은 {scale_match[1]}도부터 {scale_match[2]}도까지 표시돼 있습니다. 이 설명은 표의 눈금 범위입니다.'
            return supported_response(parent, answer, descriptions["I1.jpg"], "human_checked_image_description")
        if ("점도" in normalized_question or "0w20" in normalized_question) and oil_match:
            answer = f'표의 1.6 T-GDi HEV 행에는 {oil_match[1]}이 표시돼 있습니다.'
            return supported_response(parent, answer, descriptions["I2.jpg"], "human_checked_image_description")

    elif parent["metadata"]["record_id"] == "seatbelt_54":
        for rule in prose_rules:
            if normalize_keyword(rule["subject"]) not in normalized_question:
                continue
            if not fits_vocabulary(question, grammar_words + rule["words"]):
                continue
            match = re.search(rule["pattern"], parent["content"])
            if match is None:
                continue
            evidence = match.group()
            warning = parent["content"].split("[경고]", 1)[1].strip() if "[경고]" in parent["content"] else ""
            return supported_response(parent, "원문에서는 다음과 같이 설명합니다.\n" + evidence,
                                      evidence, "exact_pdf_sentence", warning)
    return None

def answer_manual(question):
    """검색한 원문에서 지원하는 답을 확인하거나, 현재 자료로 확인하지 못했다고 안내합니다."""
    hits = hybrid_search(question, top_k=3)
    for hit in hits:
        response = answer_from_parent(question, hit["parent_record"])
        if response is not None:
            response["matched_parent_id"] = hit["parent_record"]["metadata"]["record_id"]
            return response

    # 부품이 일치하면 상세 설명을 찾을 참조 쪽수만 안내합니다.
    hints = []
    for hit in hits:
        parent = hit["parent_record"]
        for item in parent["metadata"].get("navigation_items", []):
            if normalize_keyword(item["name"]) in normalize_keyword(question):
                hints.append({
                    "name": item["name"],
                    "target_manual_pages": item["target_manual_pages"],
                    "source": citation(parent),
                })
    return {
        "status": "not_confirmed",
        "answer": "현재 자료에서 질문의 답을 확인하기 어렵습니다.",
        "reason": "현재 원문과 지원하는 질문 규칙으로 답을 확인하지 못했습니다. 전체 PDF에 답이 없다는 뜻은 아닙니다.",
        "evidence_text": "",
        "evidence_kind": None,
        "warning_text": "",
        "sources": [],
        "image_parts": [],
        "reference_hints": hints,
    }

print("답변 상태:", "supported / not_confirmed")
print("supported도 이번에 정한 질문 유형 안에서만 사용합니다.")

답변 상태: supported / not_confirmed
supported도 이번에 정한 질문 유형 안에서만 사용합니다.


### 4. 확인 가능한 질문과 확인하지 못한 질문 점검

답이 있는 질문뿐 아니라, 같은 부품 이름으로 다른 정보를 묻는 질문도 확인합니다. 안내도에 '시동 버튼'이 있다고 해서 가격이나 교체 주기까지 답하면 안 됩니다. 또한 두 가지 정보를 한꺼번에 묻는 질문의 일부만 답하지 않도록 확인합니다.

In [4]:
# [프로젝트 추가 기록] 검색 근거를 확인한 제한된 질문에만 정해 둔 문구로 답하는 개인 규칙 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
answer_cases = [
    ("패들 쉬프트는 안내도 몇 번인가요?", "supported"),
    ("시동 버튼 위치를 알려줘.", "supported"),
    ("1.6 T-GDi HEV의 엔진 오일 SAE 점도는 무엇인가요?", "supported"),
    ("엔진 오일 점도 표의 온도 눈금 범위를 알려줘.", "supported"),
    ("안전벨트 스토퍼를 쓰면 어떤 문제가 생기나요?", "supported"),
    ("차단 클립을 버클에 꽂으면 경고등과 경고음은 어떻게 되나요?", "supported"),
    ("패들 쉬프트 사용 방법을 알려줘.", "not_confirmed"),
    ("시동 버튼 교체 가격은 얼마인가요?", "not_confirmed"),
    ("시동 버튼 위치와 서울 날씨를 알려줘.", "not_confirmed"),
    ("영하 30도에서 0W-20 엔진 오일을 사용해도 되나요?", "not_confirmed"),
    ("엔진 오일 점도와 권장 타이어 공기압을 알려줘.", "not_confirmed"),
    ("차단 클립으로 경고음을 끄는 방법을 알려줘.", "not_confirmed"),
    ("차대번호는 어디에 새겨져 있나요?", "not_confirmed"),
    ("오늘 서울 날씨를 알려줘.", "not_confirmed"),
    ("차단 클립은 어디 있나요?", "not_confirmed"),
    ("안전벨트 스토퍼는 매뉴얼 몇 쪽인가요?", "not_confirmed"),
    ("엔진 오일 점도 표의 온도는 매뉴얼 몇 쪽인가요?", "not_confirmed"),
    ("엔진 오일 점도는 몇 번인가요?", "not_confirmed"),
    ("패들 쉬프트", "not_confirmed"),
]
case_results = []
for question, expected_status in answer_cases:
    response = answer_manual(question)
    case_results.append({"question": question, "expected": expected_status, "response": response})
    print(response["status"], "|", question)

mismatches = [r for r in case_results if r["response"]["status"] != r["expected"]]
assert not mismatches, [(r["question"], r["response"]["status"]) for r in mismatches]

# 확인하지 못한 답에 근거 출처나 이미지를 잘못 붙이지 않았는지 확인합니다.
for result in case_results:
    response = result["response"]
    if response["status"] == "not_confirmed":
        assert not response["sources"] and not response["evidence_text"]
    else:
        parent = parent_lookup[response["matched_parent_id"]]
        assert response["sources"][0]["record_id"] == parent["metadata"]["record_id"]
        if response["evidence_kind"] == "exact_pdf_sentence":
            assert response["evidence_text"] in parent["content"]

print("\n확인한 질문 수:", len(case_results))
print("기대 상태와 일치:", len(case_results) - len(mismatches))
print("이 검사는 이번 규칙과 예제에 대한 확인입니다. 전체 답변 정확도가 아닙니다.")

supported | 패들 쉬프트는 안내도 몇 번인가요?
supported | 시동 버튼 위치를 알려줘.
supported | 1.6 T-GDi HEV의 엔진 오일 SAE 점도는 무엇인가요?
supported | 엔진 오일 점도 표의 온도 눈금 범위를 알려줘.
supported | 안전벨트 스토퍼를 쓰면 어떤 문제가 생기나요?
supported | 차단 클립을 버클에 꽂으면 경고등과 경고음은 어떻게 되나요?
not_confirmed | 패들 쉬프트 사용 방법을 알려줘.
not_confirmed | 시동 버튼 교체 가격은 얼마인가요?
not_confirmed | 시동 버튼 위치와 서울 날씨를 알려줘.
not_confirmed | 영하 30도에서 0W-20 엔진 오일을 사용해도 되나요?
not_confirmed | 엔진 오일 점도와 권장 타이어 공기압을 알려줘.
not_confirmed | 차단 클립으로 경고음을 끄는 방법을 알려줘.
not_confirmed | 차대번호는 어디에 새겨져 있나요?
not_confirmed | 오늘 서울 날씨를 알려줘.
not_confirmed | 차단 클립은 어디 있나요?
not_confirmed | 안전벨트 스토퍼는 매뉴얼 몇 쪽인가요?
not_confirmed | 엔진 오일 점도 표의 온도는 매뉴얼 몇 쪽인가요?
not_confirmed | 엔진 오일 점도는 몇 번인가요?
not_confirmed | 패들 쉬프트

확인한 질문 수: 19
기대 상태와 일치: 19
이 검사는 이번 규칙과 예제에 대한 확인입니다. 전체 답변 정확도가 아닙니다.


### 5. 답변 형태 살펴보기

출처는 PDF 쪽수와 매뉴얼 쪽수를 따로 보여 줍니다. 그림은 실제 이미지 파일이나 PDF에서 읽을 수 있는 참조 정보로 연결합니다. 확인하지 못한 조작법 질문에는 확정 답변 대신 원문에 있는 참조 쪽수만 별도로 안내할 수 있습니다.

In [5]:
# [프로젝트 추가 기록] 검색 근거를 확인한 제한된 질문에만 정해 둔 문구로 답하는 개인 규칙 실험입니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
for question in [
    "패들 쉬프트는 안내도 몇 번인가요?",
    "1.6 T-GDi HEV의 엔진 오일 SAE 점도는 무엇인가요?",
    "안전벨트 스토퍼를 쓰면 어떤 문제가 생기나요?",
    "패들 쉬프트 사용 방법을 알려줘.",
]:
    response = answer_manual(question)
    print("\n질문:", question)
    print("상태:", response["status"])
    print("답변:", response["answer"])
    print("출처:", response["sources"])
    print("그림:", [(p["pdf_page_number"], p["name"]) for p in response["image_parts"]])
    if response["warning_text"]:
        print("[함께 확인할 경고 원문]", response["warning_text"])
    if response["reference_hints"]:
        print("추가 확인할 매뉴얼 항목:", response["reference_hints"])
print("\n저장:", "메모리만 사용, 파일·Supabase 저장 없음")


질문: 패들 쉬프트는 안내도 몇 번인가요?
상태: supported
답변: 차량 내부 III 안내도의 패들 쉬프트 항목은 7번입니다. 항목의 참조 쪽수는 매뉴얼 395쪽입니다.
출처: [{'record_id': 'overview_35', 'title': '차량 내부 III', 'pdf_page_numbers': [35, 36], 'manual_page_number': 30, 'manual_page_numbers': [30, 31]}]
그림: [(35, 'I1.jpg')]

질문: 1.6 T-GDi HEV의 엔진 오일 SAE 점도는 무엇인가요?
상태: supported
답변: 표의 1.6 T-GDi HEV 행에는 0W-20이 표시돼 있습니다.
출처: [{'record_id': 'table_44', 'title': '온도에 따른 엔진 오일 SAE 점도 분류표', 'pdf_page_numbers': [44], 'manual_page_number': 39, 'manual_page_numbers': [39]}]
그림: [(44, 'I1.jpg'), (44, 'I2.jpg')]

질문: 안전벨트 스토퍼를 쓰면 어떤 문제가 생기나요?
상태: supported
답변: 원문에서는 다음과 같이 설명합니다.
안전벨트 스토퍼: 스토퍼를 벨트에 장착하게 되면 벨트가 정상적으로 감기는 것을 방해하여 안 전벨트 성능이 떨어집니다.
출처: [{'record_id': 'seatbelt_54', 'title': '안전벨트 착용', 'pdf_page_numbers': [54], 'manual_page_number': 49, 'manual_page_numbers': [49]}]
그림: [(54, 'I1.jpg')]
[함께 확인할 경고 원문] 안전벨트의 정상적인 착용 및 사용을 방해하는 물품들은 절대로 사용하지 마십시오. 차단 클립: 차단 클립을 버클에 꽂게 되면 안전벨트를 착용한 것으로 인식하여 안전벨트 미착용 경고등 및 경고음이 작동하지 않습니다. 안전벨트 스토퍼: 스토퍼를 벨트에 장착하게 

### 이번 실행 결과와 적용 범위

19개 예제 질문에서 기대한 상태와 모두 일치했습니다. 확인한 여섯 질문은 자료와 출처를 제공했고, 나머지 열세 질문은 보류했습니다. 이것은 제한된 답변 규칙의 동작 확인이며, 전체 PDF의 답변 정확도를 측정한 결과는 아닙니다.

지원 표현 밖의 질문은 원문에 답이 있어도 보류될 수 있습니다. 특히 현재 자료에 들어 있지 않은 차대번호를 전체 매뉴얼에도 없다고 판단하지 않습니다. 안내도의 참조 쪽수는 상세 설명의 위치를 안내할 뿐, 아직 그 상세 페이지를 읽어서 답한 것은 아닙니다.

샘플 자료에 대한 청킹·임베딩·검색·제한된 답변 흐름을 확인했습니다. 전체 PDF로 범위를 넓히는 청킹 작업과 Python 파일 정리, Supabase 저장은 별도 단계로 남아 있습니다.